# 12.15 Geometric deep learning & equivariance

Equivariant models respect the symmetries of geometric data. Geometric deep learning keeps the symmetry contract of the data: invariant outputs stay fixed and equivariant outputs transform with the input. This walkthrough checks rotations, translations, permutations, and radial messages in NumPy. Save a copy to Drive to edit.

In [ ]:

import math
import random
from collections import Counter
from collections import defaultdict

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np

SEED = 1215
random.seed(SEED)
np.random.seed(SEED)


def make_d1_graph():
    graph = nx.Graph()
    graph.add_nodes_from(range(4))
    graph.add_edges_from([(0, 1), (0, 2), (1, 2), (2, 3)])
    labels = {0: 0, 1: 0, 2: 1, 3: 1}
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_karate_rung():
    graph = nx.karate_club_graph()
    labels = {}
    for node, data in graph.nodes(data=True):
        labels[node] = 0 if data.get("club") == "Mr. Hi" else 1
    positions = nx.spring_layout(graph, seed=SEED)
    return graph, labels, positions


def make_sbm_rung(sizes, p_in, p_out, seed, noise_edges=0):
    probs = []
    for row in range(len(sizes)):
        prob_row = []
        for col in range(len(sizes)):
            prob_row.append(p_in if row == col else p_out)
        probs.append(prob_row)
    graph = nx.stochastic_block_model(sizes, probs, seed=seed)
    rng = np.random.default_rng(seed)
    nodes = list(graph.nodes())
    for _ in range(noise_edges):
        u = int(rng.choice(nodes))
        v = int(rng.choice(nodes))
        if u != v:
            graph.add_edge(u, v)
    labels = {}
    offset = 0
    for block, size in enumerate(sizes):
        for node in range(offset, offset + size):
            labels[node] = block
        offset = offset + size
    positions = nx.spring_layout(graph, seed=seed)
    return graph, labels, positions


def make_cora_like_rung():
    graph, labels, positions = make_sbm_rung([18, 16, 14], 0.26, 0.035, SEED + 4, noise_edges=8)
    rng = np.random.default_rng(SEED + 44)
    years = {}
    node_types = {}
    for node in graph.nodes():
        years[node] = int(2016 + rng.integers(0, 7))
        node_types[node] = "paper"
    nx.set_node_attributes(graph, years, "year")
    nx.set_node_attributes(graph, node_types, "kind")
    return graph, labels, positions


def make_large_noisy_rung():
    graph, labels, positions = make_sbm_rung([35, 32, 30, 28], 0.18, 0.045, SEED + 5, noise_edges=55)
    rng = np.random.default_rng(SEED + 55)
    for u, v in graph.edges():
        graph[u][v]["time"] = int(rng.integers(0, 10))
        graph[u][v]["relation"] = "strong" if labels[u] == labels[v] else "weak"
    return graph, labels, positions


def build_graph_ladder():
    d1_graph, d1_labels, d1_pos = make_d1_graph()
    d2_graph, d2_labels, d2_pos = make_karate_rung()
    d3_graph, d3_labels, d3_pos = make_sbm_rung([12, 12, 12], 0.32, 0.04, SEED + 3, noise_edges=5)
    d4_graph, d4_labels, d4_pos = make_cora_like_rung()
    d5_graph, d5_labels, d5_pos = make_large_noisy_rung()
    return [
        {"name": "D1 toy", "graph": d1_graph, "labels": d1_labels, "pos": d1_pos},
        {"name": "D2 karate", "graph": d2_graph, "labels": d2_labels, "pos": d2_pos},
        {"name": "D3 SBM", "graph": d3_graph, "labels": d3_labels, "pos": d3_pos},
        {"name": "D4 Cora-like subset", "graph": d4_graph, "labels": d4_labels, "pos": d4_pos},
        {"name": "D5 larger noisy", "graph": d5_graph, "labels": d5_labels, "pos": d5_pos},
    ]


def partition_from_labels(labels):
    groups = defaultdict(set)
    for node, label in labels.items():
        groups[label].add(node)
    return list(groups.values())


def accuracy_score(y_true, y_pred):
    correct = 0
    total = 0
    for node, truth in y_true.items():
        if node in y_pred:
            correct = correct + int(y_pred[node] == truth)
            total = total + 1
    return correct / max(total, 1)


## The concept, built once (D1): invariance and equivariance

A map is invariant when $f(gx)=f(x)$ and equivariant when $f(gx)=g f(x)$. The lesson checks distance $1.414$, residual $[0,0]$, relative edge $[2,1]$, permutation identity $(PAP^\top)(PX)=P(AX)$, and radial message rotation.

In [ ]:

def rotation_matrix(theta):
    return np.array(
        [
            [math.cos(theta), -math.sin(theta)],
            [math.sin(theta), math.cos(theta)],
        ]
    )


def radial_messages(points, edges):
    messages = np.zeros_like(points)
    for source, target in edges:
        relative = points[target] - points[source]
        radius = np.linalg.norm(relative) + 1e-9
        messages[source] = messages[source] + relative / radius
    return messages


def equivariant_message(adjacency, features):
    return adjacency @ features


## Check rotations, translations, permutations, and radial messages

Distances are invariant to rotation, vectors rotate equivariantly, and relative edge messages preserve translation symmetry.

In [ ]:

points = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [3.0, 1.0],
    ]
)
rot = rotation_matrix(math.pi / 2)
rotated = points @ rot.T
distance = np.linalg.norm(points[0] - points[1])
rotated_distance = np.linalg.norm(rotated[0] - rotated[1])
commute_residual = rot @ (3.0 * points[0]) - 3.0 * (rot @ points[0])
edge_vector = points[2] - points[0]
shift = np.array([10.0, -4.0])
shifted_edge = (points[2] + shift) - (points[0] + shift)
adjacency = np.array(
    [
        [0.0, 1.0, 0.0],
        [1.0, 0.0, 1.0],
        [0.0, 1.0, 0.0],
    ]
)
features = np.array(
    [
        [1.0, 0.0],
        [0.0, 1.0],
        [2.0, 1.0],
    ]
)
permutation = np.array(
    [
        [0.0, 1.0, 0.0],
        [1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0],
    ]
)
left = (permutation @ adjacency @ permutation.T) @ (permutation @ features)
right = permutation @ (adjacency @ features)
edges = [(0, 1), (1, 2), (2, 0)]
message = radial_messages(points, edges)
rotated_message = radial_messages(rotated, edges)
expected_rotated_message = message @ rot.T
print("distance", round(distance, 3), round(rotated_distance, 3))
print("rotation commute residual", commute_residual.round(6).tolist())
print("translated edge", shifted_edge.tolist())
print("permutation identity", np.allclose(left, right))
print("radial message rotates", np.allclose(rotated_message, expected_rotated_message))
assert round(distance, 3) == 1.414
assert round(rotated_distance, 3) == 1.414
assert np.allclose(commute_residual, np.array([0.0, 0.0]))
assert edge_vector.tolist() == [2.0, 1.0]
assert shifted_edge.tolist() == [2.0, 1.0]
assert np.allclose(left, right)
assert np.allclose(rotated_message, expected_rotated_message)


## The dataset ladder

D1 is a geometric toy; D2 karate; D3 noisy SBM; D4 Cora-like subset; D5 larger noisy graph with layout coordinates.

In [ ]:

ladder = build_graph_ladder()
for rung in ladder:
    graph = rung["graph"]
    labels = rung["labels"]
    sample_nodes = list(graph.nodes())[:5]
    sample_edges = list(graph.edges())[:5]
    print(rung["name"])
    print("  nodes", graph.number_of_nodes(), "edges", graph.number_of_edges(), "classes", sorted(set(labels.values())))
    print("  sample nodes", sample_nodes)
    print("  sample edges", sample_edges)


## Run the same method across D1-D5

In [ ]:

def coordinates_from_layout(rung):
    graph = rung["graph"]
    positions = rung["pos"]
    coords = np.array([positions[node] for node in graph.nodes()])
    return coords


def relative_distance_features(graph, coords):
    node_list = list(graph.nodes())
    index = {node: idx for idx, node in enumerate(node_list)}
    features = []
    for node in node_list:
        distances = []
        for neighbor in graph.neighbors(node):
            distances.append(np.linalg.norm(coords[index[neighbor]] - coords[index[node]]))
        if distances:
            features.append([np.mean(distances), len(distances)])
        else:
            features.append([0.0, 0.0])
    return np.array(features)


def label_by_nearest_centroid(features, labels, nodes):
    centroids = {}
    for label in sorted(set(labels.values())):
        rows = [features[index] for index, node in enumerate(nodes) if labels[node] == label]
        centroids[label] = np.mean(rows, axis=0)
    predictions = {}
    for index, node in enumerate(nodes):
        distances = {label: np.linalg.norm(features[index] - center) for label, center in centroids.items()}
        predictions[node] = min(distances, key=distances.get)
    return predictions


ladder = build_graph_ladder()
equiv_results = []
for rung in ladder:
    graph = rung["graph"]
    nodes = list(graph.nodes())
    coords = coordinates_from_layout(rung)
    features = relative_distance_features(graph, coords)
    rotated_coords = coords @ rotation_matrix(math.pi / 3).T
    rotated_features = relative_distance_features(graph, rotated_coords)
    predictions = label_by_nearest_centroid(rotated_features, rung["labels"], nodes)
    score = accuracy_score(rung["labels"], predictions)
    equiv_error = float(np.max(np.abs(features - rotated_features)))
    rung["coords"] = coords
    rung["rotated_coords"] = rotated_coords
    rung["predictions"] = predictions
    rung["accuracy"] = score
    rung["equiv_error"] = equiv_error
    equiv_results.append(score)
    print(f"{rung['name']}: transformed_accuracy={score:.3f} invariant_feature_error={equiv_error:.6f}")


## Results visualization

Top row: transformed graph panels colored by predictions. Bottom left: node accuracy under rotation.

In [ ]:

fig, axes = plt.subplots(2, 5, figsize=(18, 7))
for index, rung in enumerate(ladder):
    graph = rung["graph"]
    nodes = list(graph.nodes())
    rotated_pos = {node: rung["rotated_coords"][idx] for idx, node in enumerate(nodes)}
    colors = [rung["predictions"][node] for node in graph.nodes()]
    nx.draw_networkx(
        graph,
        pos=rotated_pos,
        node_color=colors,
        cmap="tab10",
        with_labels=False,
        node_size=55,
        edge_color="#cccccc",
        ax=axes[0, index],
    )
    axes[0, index].set_title(rung["name"])
    axes[0, index].axis("off")
axes[1, 0].plot(range(1, 6), equiv_results, marker="o")
axes[1, 0].set_xticks(range(1, 6))
axes[1, 0].set_xlabel("rung")
axes[1, 0].set_ylabel("node accuracy")
axes[1, 0].set_title("accuracy after rotation")
for axis in axes[1, 1:]:
    axis.axis("off")
plt.tight_layout()


## Pitfall on the hardest rung: absolute coordinates break translation symmetry

Raw coordinates move under translation; relative distances and edge vectors do not.

In [ ]:

rung = ladder[-1]
coords = rung["coords"]
shift = np.array([8.0, -3.0])
absolute_before = coords.mean(axis=1)
absolute_after = (coords + shift).mean(axis=1)
relative_before = relative_distance_features(rung["graph"], coords)
relative_after = relative_distance_features(rung["graph"], coords + shift)
absolute_change = float(np.mean(np.abs(absolute_after - absolute_before)))
relative_change = float(np.max(np.abs(relative_after - relative_before)))
print("absolute-coordinate feature shift", round(absolute_change, 3))
print("relative-distance feature shift", round(relative_change, 6))
print("Fix: build messages from relative vectors or distances, not raw absolute positions.")



## Evaluate it + practice

- Metric: compare the rung's node accuracy under transformed inputs against the no-skill baseline `majority class` on D1-D5.
- Sanity check: rerun with the fixed seed and verify D1 reproduces the asserted lesson numbers before trusting larger rungs.
- Ablation: replace relative features with absolute coordinates after a large shift; the metric should drop or the failure should become visible.
- Failure signal: a high score with a violated split, symmetry, or relation contract is not a real win.
- Inspect the hardest rung by plotting both the graph artifact and the metric curve rather than reading one scalar alone.

Practice prompts:
1. Change only the D3 noise level and predict how the metric curve should move.


2. Replace the D5 seed with another fixed seed and compare the artifact panel.

3. Add one cheap assertion that would catch the lesson pitfall before the metric is printed.